# Food Delivery Time Prediction
Binary Classification using Naive Bayes, KNN, and Decision Tree


In [7]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score, roc_curve, auc
)
df = pd.read_csv(r"C:\Users\DELL\Desktop\Machine Learning\Food_Delivery_Time_Prediction1.csv")
df.head()

,Order_ID,Customer_Location,Restaurant_Location,Distance,Weather_Conditions,Traffic_Conditions,Delivery_Person_Experience,Order_Priority,Order_Time,Vehicle_Type,Restaurant_Rating,Customer_Rating,Delivery_Time,Order_Cost,Tip_Amount
0,ORD0001,"(17.030479, 79.743077)","(12.358515, 85.100083)",1.57,Rainy,Medium,4,Medium,Afternoon,Car,4.1,3.0,26.22,1321.10,81.54
1,ORD0002,"(15.398319, 86.639122)","(14.174874, 77.025606)",21.32,Cloudy,Medium,8,Low,Night,Car,4.5,4.2,62.61,152.21,29.02
2,ORD0003,"(15.687342, 83.888808)","(19.594748, 82.048482)",6.95,Snowy,Medium,9,High,Night,Bike,3.3,3.4,48.43,1644.38,64.17
3,ORD0004,"(20.415599, 78.046984)","(16.915906, 78.278698)",13.79,Cloudy,Low,2,Medium,Evening,Bike,3.2,3.7,111.63,541.25,79.23
4,ORD0005,"(14.786904, 78.706532)","(15.206038, 86.203182)",6.72,Rainy,High,6,Low,Night,Bike,3.5,2.8,32.38,619.81,2.34


In [8]:

df.info()
df.isnull().sum()


<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Order_ID                    200 non-null    str    
 1   Customer_Location           200 non-null    str    
 2   Restaurant_Location         200 non-null    str    
 3   Distance                    200 non-null    float64
 4   Weather_Conditions          200 non-null    str    
 5   Traffic_Conditions          200 non-null    str    
 6   Delivery_Person_Experience  200 non-null    int64  
 7   Order_Priority              200 non-null    str    
 8   Order_Time                  200 non-null    str    
 9   Vehicle_Type                200 non-null    str    
 10  Restaurant_Rating           200 non-null    float64
 11  Customer_Rating             200 non-null    float64
 12  Delivery_Time               200 non-null    float64
 13  Order_Cost                  200 non-null    fl

Order_ID                      0
Customer_Location             0
Restaurant_Location           0
Distance                      0
Weather_Conditions            0
Traffic_Conditions            0
Delivery_Person_Experience    0
Order_Priority                0
Order_Time                    0
Vehicle_Type                  0
Restaurant_Rating             0
Customer_Rating               0
Delivery_Time                 0
Order_Cost                    0
Tip_Amount                    0
dtype: int64

In [9]:

import ast

def extract_lat_lon(loc):
    lat, lon = ast.literal_eval(loc)
    return lat, lon

df[['cust_lat','cust_lon']] = df['Customer_Location'].apply(lambda x: pd.Series(extract_lat_lon(x)))
df[['rest_lat','rest_lon']] = df['Restaurant_Location'].apply(lambda x: pd.Series(extract_lat_lon(x)))


In [10]:

from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2-lat1)
    dlon = radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R*c

df['Geo_Distance'] = df.apply(
    lambda row: haversine(
        row['cust_lat'], row['cust_lon'],
        row['rest_lat'], row['rest_lon']
    ),
    axis=1
)

threshold = df['Delivery_Time'].median()
df['Delivery_Status'] = np.where(df['Delivery_Time'] > threshold, 1, 0)


In [ ]:

categorical_cols = [
    'Weather_Conditions',
    'Traffic_Conditions',
    'Order_Priority',
    'Order_Time',
    'Vehicle_Type'
]

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

numeric_cols = df.select_dtypes(include=np.number).columns

imputer = SimpleImputer(strategy='median')
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])


In [12]:

drop_cols = [
    'Order_ID',
    'Customer_Location',
    'Restaurant_Location',
    'Delivery_Time',
    'Delivery_Status'
]

X = df.drop(columns=drop_cols)
y = df['Delivery_Status']

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


ValueError: could not convert string to float: 'Afternoon'

In [ ]:

def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)

    print(f"\n{model_name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

    print(classification_report(y_test, y_pred))

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    }


In [ ]:

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
nb_results = evaluate_model(nb_model, X_test, y_test, "Naive Bayes")


In [ ]:

knn_params = {'n_neighbors':[3,5,7,9,11]}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    knn_params,
    cv=5,
    scoring='accuracy'
)

knn_grid.fit(X_train, y_train)

knn_model = knn_grid.best_estimator_
print("Best K:", knn_grid.best_params_)

knn_results = evaluate_model(knn_model, X_test, y_test, "KNN")


In [ ]:

dt_params = {
    'max_depth':[3,5,7,10,None],
    'min_samples_split':[2,5,10]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params,
    cv=5,
    scoring='accuracy'
)

dt_grid.fit(X_train, y_train)

dt_model = dt_grid.best_estimator_
print("Best Parameters:", dt_grid.best_params_)

dt_results = evaluate_model(dt_model, X_test, y_test, "Decision Tree")


In [ ]:

results = pd.DataFrame([nb_results, knn_results, dt_results])
results.sort_values(by="Accuracy", ascending=False)


In [ ]:

models = {
    "Naive Bayes": nb_model,
    "KNN": knn_model,
    "Decision Tree": dt_model
}

plt.figure(figsize=(8,6))

for name, model in models.items():
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_test)[:,1]
        fpr, tpr, _ = roc_curve(y_test, probs)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.2f})")

plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()
